# Chatbot Graph Visualization

This notebook renders the current chatbot orchestration graph from the application code.

- `get_graph()` shows the top-level single-agent spending coach runtime.
- `get_graph(xray=True)` expands the local `create_agent` runnable graph used by this notebook's fake model setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "lab":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from typing import Any, Sequence

from IPython.display import Image, Markdown, display
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import BaseTool

from app.domain.chatbot.agents.graph import ChatbotToolBundle, build_chatbot_graph


class BindableFakeChatModel(FakeMessagesListChatModel):
    def bind_tools(
        self,
        tools: Sequence[BaseTool | dict | type | Any],
        *,
        tool_choice: str | None = None,
        **kwargs: Any,
    ) -> "BindableFakeChatModel":
        return self


def build_fake_model(_: str) -> BindableFakeChatModel:
    return BindableFakeChatModel(responses=[AIMessage(content="visualization")])


In [ ]:
chatbot_graph = build_chatbot_graph(
    model_factory=build_fake_model,
    tools=ChatbotToolBundle.empty(),
)

compiled_graph = chatbot_graph.get_graph()
expanded_graph = chatbot_graph.get_graph(xray=True)


In [ ]:
display(Markdown("## Top-level graph"))
display(Markdown(f"```mermaid\n{compiled_graph.draw_mermaid()}\n```"))


In [ ]:
display(Markdown("## Expanded graph with subgraphs"))
display(Markdown(f"```mermaid\n{expanded_graph.draw_mermaid()}\n```"))


In [ ]:
Image(expanded_graph.draw_mermaid_png())
